# 10 — Asymptotic Theory
**References:** van der Vaart (1998) Ch. 3, 5–7 · Wasserman (2004) Ch. 9 · Efron & Hastie (2016) Ch. 4

## Narrative thread
```
Delta method -> Influence functions -> Asymptotic relative efficiency -> Parametric vs semiparametric
```

## 1. Delta method (revisited and extended)

The delta method (introduced in notebook 04) propagates asymptotic distributions through smooth maps.

**Second-order delta method:** When $g'(\theta) = 0$:
$$n(g(\hat\theta) - g(\theta)) \xrightarrow{d} \frac{g''(\theta)}{2} \cdot \sigma^2 \chi^2(1)$$

**Multivariate delta method:** If $\sqrt{n}(\hat{\boldsymbol{\theta}} - \boldsymbol{\theta}) \xrightarrow{d} \mathcal{N}(\mathbf{0}, \boldsymbol{\Sigma})$:
$$\sqrt{n}(g(\hat{\boldsymbol{\theta}}) - g(\boldsymbol{\theta})) \xrightarrow{d} \mathcal{N}(0, \nabla g^\top \boldsymbol{\Sigma}\, \nabla g)$$

**Variance-stabilizing transformations:** Choose $g$ such that $[g'(\theta)]^2 \text{Var}(X) = $ const.

| Distribution | Parameter | VST | Stabilized variance |
|---|---|---|---|
| Binomial($n,p$) | $p$ | $g(p) = \arcsin(\sqrt{p})$ | $1/(4n)$ |
| Poisson($\lambda$) | $\lambda$ | $g(\lambda) = \sqrt{\lambda}$ | $1/4$ |
| Normal($\mu,\sigma^2$) | $\sigma$ | $g(\sigma) = \log\sigma$ | $1/(2n)$ |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

# ── VST: arcsin for Binomial proportion ──────────────────────────────────
# Var(p_hat) = p(1-p)/n  -- depends on p
# Var(arcsin(sqrt(p_hat))) ≈ 1/(4n)  -- constant

n_obs  = 50
n_reps = 20_000
ps_test = [0.05, 0.3, 0.5]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for col, p_true in enumerate(ps_test):
    p_hats = np.random.binomial(n_obs, p_true, n_reps) / n_obs
    vst    = np.arcsin(np.sqrt(p_hats))
    vst_mean = np.arcsin(np.sqrt(p_true))

    # Raw proportion
    ax0 = axes[0, col]
    ax0.hist(p_hats, bins=40, density=True, color='#4361ee', alpha=0.7, edgecolor='white')
    ax0.set_title(f'$\\hat p$, p={p_true}\nVar={p_hats.var():.5f} (expected {p_true*(1-p_true)/n_obs:.5f})')

    # VST
    ax1 = axes[1, col]
    # Standardize using known asymptotic var = 1/(4n)
    z_vst = (vst - vst_mean) / np.sqrt(1/(4*n_obs))
    ax1.hist(z_vst, bins=50, density=True, color='#f72585', alpha=0.7, edgecolor='white')
    x_std = np.linspace(-4, 4, 300)
    ax1.plot(x_std, stats.norm.pdf(x_std), 'k-', lw=2)
    ax1.set_title(f'VST $\\arcsin(\\sqrt{{\\hat p}})$ standardized, p={p_true}\n'
                   f'Var of VST={vst.var():.5f} (expected {1/(4*n_obs):.5f})')

axes[0, 1].set_title(axes[0,1].get_title())  # keep
plt.suptitle('Variance-Stabilizing Transform: arcsin(sqrt(p_hat))\n'
             'Row 1: raw proportion (variance depends on p); Row 2: VST (variance ≈ 1/4n)',
             fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Influence functions

The **influence function (IF)** of an estimator $T$ at distribution $F$ measures the effect of an infinitesimal contamination at point $x$:

$$\text{IF}(x; T, F) = \lim_{\varepsilon \to 0} \frac{T((1-\varepsilon)F + \varepsilon\delta_x) - T(F)}{\varepsilon}$$

where $\delta_x$ is a point mass at $x$.

**Key results:**
- The asymptotic variance of $\sqrt{n}(T(F_n) - T(F))$ is $\int \text{IF}(x; T, F)^2\,dF(x)$
- For the mean: $\text{IF}(x; \bar X, F) = x - \mu$ — unbounded, so mean is **not robust**
- For the median: $\text{IF}(x; \text{med}, F) = \text{sgn}(x-m)/(2f(m))$ — bounded, so median is **robust**

**Gross error sensitivity:** $\kappa^* = \sup_x |\text{IF}(x)|$ — bounded $\kappa^*$ means robustness.

In [ ]:
# ── Influence of a single outlier on mean vs median ───────────────────────
n_base   = 30
base_data = np.random.normal(0, 1, n_base)
outlier_vals = np.linspace(-5, 30, 200)

mean_with_outlier   = [(np.append(base_data, v).mean()   - base_data.mean())   * (n_base+1) for v in outlier_vals]
median_with_outlier = [(np.median(np.append(base_data, v)) - np.median(base_data)) * (n_base+1) for v in outlier_vals]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(outlier_vals, mean_with_outlier,   color='#f72585', lw=2.5, label='Mean IF (unbounded)')
ax.plot(outlier_vals, median_with_outlier, color='#4361ee', lw=2.5, label='Median IF (bounded)')
ax.set_xlabel('Outlier value $x$')
ax.set_ylabel('$(n+1)\\cdot(T(x_1,\\ldots,x_n,x) - T(x_1,\\ldots,x_n))$')
ax.set_title('Influence function: mean vs median\nMean influenced proportionally; median bounded')
ax.legend()

# ── Asymptotic Relative Efficiency: Normal location ───────────────────────
# ARE(median, mean) for Normal = pi/2 ≈ 0.637 -- mean is 57% more efficient
# ARE(median, mean) for Laplace(0,1) = 2 -- median is 2x more efficient

n_reps_are = 10_000
ns_are     = [20, 50, 100, 200]

for dist_name, sampler in [('Normal(0,1)', lambda n: np.random.normal(0,1,n)),
                             ('Laplace(0,1)', lambda n: np.random.laplace(0,1,n))]:
    are_empirical = []
    for n_ in ns_are:
        means_var  = np.var([sampler(n_).mean()   for _ in range(n_reps_are)])
        medians_var = np.var([np.median(sampler(n_)) for _ in range(n_reps_are)])
        are_empirical.append(means_var / medians_var)  # ARE(median, mean) = Var(mean)/Var(median)
    axes[1].plot(ns_are, are_empirical, 'o-', lw=2, markersize=6,
                 label=f'{dist_name} (ARE≈{are_empirical[-1]:.2f})')

axes[1].axhline(np.pi/2, color='gray', lw=1.5, linestyle='--', label=f'π/2={np.pi/2:.2f} (Normal, theoretical)')
axes[1].axhline(2.0, color='gray', lw=1.5, linestyle=':', label='2.0 (Laplace, theoretical)')
axes[1].set_xlabel('n')
axes[1].set_ylabel('ARE(median, mean) = Var(mean)/Var(median)')
axes[1].set_title('Asymptotic Relative Efficiency\n>1: median more efficient; <1: mean more efficient')
axes[1].legend(fontsize=8)

plt.suptitle('Influence Functions and Asymptotic Relative Efficiency', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nAre(median, mean):')
print(f'  Normal:  theoretical π/2 ≈ {np.pi/2:.4f} — mean is more efficient')
print(f'  Laplace: theoretical 2.0  — median is more efficient (heavier tails)')

## 3. Asymptotic relative efficiency (ARE)

For two consistent estimators $T_1$, $T_2$ with asymptotic variances $v_1$, $v_2$:

$$\text{ARE}(T_1, T_2) = \frac{v_2}{v_1}$$

$\text{ARE} > 1$ means $T_1$ is more efficient. The MLE achieves ARE = 1 relative to any other consistent estimator (Cramér-Rao + asymptotic normality).

**Notable AREs:**

| $T_1$ | $T_2$ | Distribution | ARE($T_1$, $T_2$) |
|---|---|---|---|
| Mean | Median | Normal | $\pi/2 \approx 1.57$ (mean better) |
| Median | Mean | Laplace | $2$ (median better) |
| Mean | Median | $t_3$ | $\approx 1.62$ (median better) |

## 4. Key takeaways

| Concept | Statement |
|---|---|
| Delta method | $\text{SE}(g(\hat\theta)) \approx |g'(\hat\theta)|\cdot\text{SE}(\hat\theta)$ |
| VST | Choose $g$ to make variance constant in $\theta$ |
| Influence function | Measures sensitivity to a single observation |
| Bounded IF $\Leftrightarrow$ robustness | Mean: not robust; median: robust |
| ARE | Relative efficiency between estimators; MLE achieves minimum |

**Next:** notebook 11 — Bayesian inference: the full posterior-based approach to uncertainty quantification.